# Telecom Churn Prediction - Model Experiments

This notebook compares a small set of candidate models for telecom churn prediction.

## Objectives
- Build a consistent preprocessing flow
- Train baseline and advanced models
- Compare model performance
- Select the final model for deployment

## Models Compared
- Logistic Regression
- Random Forest
- XGBoost

## Notes
The production training pipeline is implemented in `src/telecom_churn_prediction/ml/`.
This notebook is for experimentation and reporting.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd

from telecom_churn_prediction.data.data_loader import load_raw_dataset, clean_dataset
from telecom_churn_prediction.data.data_splitter import (
    split_features_and_target,
    split_train_test,
)
from telecom_churn_prediction.data.data_validator import validate_dataset
from telecom_churn_prediction.ml.metrics import calculate_classification_metrics
from telecom_churn_prediction.ml.model_factory import create_model
from telecom_churn_prediction.ml.preprocessing import build_preprocessing_pipeline
from telecom_churn_prediction.ml.threshold_optimization import find_best_threshold

from sklearn.pipeline import Pipeline

## Load and Prepare Data
We reuse the same preprocessing assumptions as the production pipeline.

In [2]:
df = clean_dataset(load_raw_dataset())
validate_dataset(df)

x, y = split_features_and_target(df)
x_train, x_test, y_train, y_test = split_train_test(x, y)

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

2026-04-01 11:56:39,001 | INFO | telecom_churn_prediction.data.data_loader | Loading dataset from C:\Users\Dileep Samaji\Desktop\AI\telecom-churn-prediction\data\raw\telco_customer_churn.csv


Train shape: (5625, 19)
Test shape: (1407, 19)


## Define Training Helper
This helper trains a model, applies threshold optimization, and returns evaluation metrics.

In [3]:
def run_model_experiment(model_name: str) -> dict[str, float | str]:
    preprocessing = build_preprocessing_pipeline()
    model = create_model(model_name)

    pipeline = Pipeline(
        steps=[
            ("preprocessing", preprocessing),
            ("model", model),
        ]
    )

    pipeline.fit(x_train, y_train)

    y_proba = pipeline.predict_proba(x_test)[:, 1]
    threshold, _ = find_best_threshold(y_test, y_proba)
    y_pred = (y_proba >= threshold).astype(int)

    metrics = calculate_classification_metrics(y_test, y_pred, y_proba)

    return {
        "model": model_name,
        "threshold": round(threshold, 4),
        **{k: round(v, 4) for k, v in metrics.items()},
    }

## Train Candidate Models
We evaluate three models:
- Logistic Regression as a linear baseline
- Random Forest as a bagged tree ensemble
- XGBoost as the final high-performance tabular model candidate

In [4]:
results = []
for model_name in ["logistic_regression", "random_forest", "xgboost"]:
    results.append(run_model_experiment(model_name))

results_df = pd.DataFrame(results)
results_df

,model,threshold,accuracy,precision,recall,f1_score,specificity,roc_auc
0,logistic_regression,0.65,0.7811,0.5747,0.6791,0.6225,0.8180,0.8351
1,random_forest,0.50,0.7555,0.5274,0.7727,0.6269,0.7493,0.8341
2,xgboost,0.30,0.7448,0.5146,0.7086,0.5962,0.7580,0.8146


## Model Comparison
We compare performance across:
- Accuracy
- Precision
- Recall
- F1-score
- Specificity
- ROC-AUC

In [5]:
results_df.sort_values(by="roc_auc", ascending=False)

,model,threshold,accuracy,precision,recall,f1_score,specificity,roc_auc
0,logistic_regression,0.65,0.7811,0.5747,0.6791,0.6225,0.8180,0.8351
1,random_forest,0.50,0.7555,0.5274,0.7727,0.6269,0.7493,0.8341
2,xgboost,0.30,0.7448,0.5146,0.7086,0.5962,0.7580,0.8146


## Interpretation of Results

### Logistic Regression
Provides a strong baseline and is easy to interpret, but may underfit complex nonlinear churn behavior.

### Random Forest
Captures nonlinear relationships and is more robust than a single decision tree.

### XGBoost
Usually provides the strongest performance on structured tabular data and supports feature-level explainability through SHAP.

## Final Model Choice

The final deployed model is **XGBoost**, because it offers:
- strong discriminative performance
- good handling of tabular churn data
- compatibility with SHAP-based explanation
- production suitability for a FastAPI inference service

## Conclusion

This experiment supports the final project choice:
- use XGBoost for deployment
- retain probability-based outputs
- optimize threshold for business-aware churn intervention